# Task B -- 100% of the data: TAPT on every comment, five seeds on every row

Nothing is held out, filtered or deduplicated at either stage.

| stage | `f_tapt` (last run) | this notebook |
|---|---|---|
| TAPT text | 6,044 of 6,406 comments (94.3%) | **6,406 of 6,406 (100%)** |
| classifier rows per model | 3,143 of 3,159 (99.5%) | **3,159 of 3,159 (100%)** |
| seeds, averaged | 42 43 44 45 46 | 42 43 44 45 46 |
| epochs | 6 | 6 |

TAPT reads `multiclass_train.csv` (3,159 comments) plus the external Kannada OffensEval
corpus (3,247), text only. The classifier trains on every row of `multiclass_train.csv`,
including the 12 repeated comments and the 4 rows whose copies carry different labels,
which earlier runs dropped.

**There is no local score, and there cannot be one.** Every labelled row is in training.
The only way to score this is to upload `b_tapt_100.zip` to CodaBench.

About 2 hours on a T4. Upload this file only; it clones the code itself.
Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on.
Save Version -> Save & Run All.

In [ ]:
import os, re, subprocess, sys, pathlib
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK); print("cwd:", os.getcwd())
subprocess.run(["git","log","-1","--oneline"], check=True)

# The flags that let TAPT read every comment. If this clone of task-b predates
# them, apply them here so the notebook does not depend on a push.
TAPT_PATCH = r"""diff --git a/work/tapt.py b/work/tapt.py
index 7426a49..5e55ce9 100644
--- a/work/tapt.py
+++ b/work/tapt.py
@@ -91,7 +91,12 @@ def main():
     ap.add_argument("--warmup", type=float, default=0.06)
     ap.add_argument("--val-frac", type=float, default=0.05,
                     help="held out to report MLM perplexity, which is the only honest "
-                         "signal that this pass did anything")
+                         "signal that this pass did anything. 0 trains on every comment "
+                         "and skips the perplexity report")
+    ap.add_argument("--min-words", type=int, default=2,
+                    help="drop comments shorter than this many words. 1 keeps all of them")
+    ap.add_argument("--no-dedupe", action="store_true",
+                    help="keep repeated comments instead of collapsing identical text")
     ap.add_argument("--no-demojize", action="store_true")
     ap.add_argument("--amp", choices=["auto", "off", "fp16", "bf16"], default="auto")
     ap.add_argument("--seed", type=int, default=42)
@@ -105,7 +110,8 @@ def main():
     demoji = not args.no_demojize
     texts = corpora.load((args.corpus or DEFAULT_CORPUS) + args.extra,
                          clean_fn=lambda t: clean(t, demojize=demoji),
-                         allow_transductive=args.allow_transductive)
+                         allow_transductive=args.allow_transductive,
+                         min_words=args.min_words, dedupe=not args.no_dedupe)
     if args.limit:
         texts = texts[:args.limit]
         print(f"LIMIT: {len(texts)} comments (smoke test)", flush=True)
@@ -144,9 +150,15 @@ def main():
             n += len(b["input_ids"])
         return math.exp(tot / max(n, 1))
 
+    print(f"MLM trains on {len(split['train'])} of {len(texts)} comments, "
+          f"{n_val} held out for perplexity", flush=True)
     print(f"effective batch {args.bs} x {accum} = {args.bs * accum}, "
           f"{math.ceil(len(tr) / accum)} optimizer steps/epoch", flush=True)
-    print(f"held-out perplexity before {perplexity():.2f}", flush=True)
+    # with nothing held out there is no perplexity to report, so the epoch
+    # lines show the training loss alone
+    ppl = (lambda: f" held-out perplexity {perplexity():.2f}") if n_val else (lambda: "")
+    if n_val:
+        print(f"held-out perplexity before {perplexity():.2f}", flush=True)
     opt.zero_grad(set_to_none=True)
     for ep in range(1, args.epochs + 1):
         model.train()
@@ -166,8 +178,8 @@ def main():
             scaler.update()
             sched.step()
             opt.zero_grad(set_to_none=True)
-        print(f"  epoch {ep}/{args.epochs} train loss {run_loss/len(tr):.4f} "
-              f"held-out perplexity {perplexity():.2f}", flush=True)
+        print(f"  epoch {ep}/{args.epochs} train loss {run_loss/len(tr):.4f}{ppl()}",
+              flush=True)
 
     out = ROOT / args.out
     out.mkdir(parents=True, exist_ok=True)
"""
if "--min-words" not in pathlib.Path("work/tapt.py").read_text():
    pathlib.Path("/tmp/tapt_flags.patch").write_text(TAPT_PATCH)
    subprocess.run(["git","apply","/tmp/tapt_flags.patch"], check=True)
    print("patched work/tapt.py with --min-words / --no-dedupe / --val-frac 0")
assert "--min-words" in pathlib.Path("work/tapt.py").read_text()

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    """Streams, tees, and raises. Nothing here is allowed to fail quietly."""
    print(f"$ {' '.join(cmd)}", flush=True)
    fh = open(log,"w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")

## 1. TAPT on every comment (~25 min)

`--val-frac 0` holds nothing back, `--min-words 1` keeps one-word comments and
`--no-dedupe` keeps repeated ones. The next cell reads the log and stops the notebook if
anything was left out.

In [ ]:
run([sys.executable,"-u","work/tapt.py","--out","work/runs/tapt-100",
     "--val-frac","0","--min-words","1","--no-dedupe"],
    log="work/tapt_100.log")

In [ ]:
t = pathlib.Path("work/tapt_100.log").read_text()
m = re.search(r"MLM trains on (\d+) of (\d+) comments, (\d+) held out", t)
assert m, "could not find the corpus line in the TAPT log"
used, total, held = map(int, m.groups())
print(f"TAPT trained on {used} of {total} comments, {held} held out")
assert used == total == 6406 and held == 0, "TAPT did not use all 6,406 comments"

## 2. Five seeds on every row (~95 min)

`--folds 1` trains each seed on all rows and keeps its final checkpoint. `--no-dedupe`
keeps all 3,159. The five models' test probabilities are averaged.

In [ ]:
run([sys.executable,"-u","work/muril_b.py","--tag","b_tapt_100",
     "--model","work/runs/tapt-100","--folds","1","--no-dedupe",
     "--seeds","42","43","44","45","46","--epochs","6"],
    log="work/b_tapt_100.log")

In [ ]:
t = pathlib.Path("work/b_tapt_100.log").read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", t)
for seed, rows in fits:
    print(f"seed {seed}: trained on {rows} rows")
assert [s for s, _ in fits] == ["42","43","44","45","46"], "not all five seeds ran"
assert all(r == "3159" for _, r in fits), "a seed trained on fewer than 3,159 rows"

## 3. Package

The agreement figure is how often this run's 395 predictions match `b_tapt_5f`, which
scored 0.5922 on CodaBench. It is not a score.

In [ ]:
import pandas as pd
ZIP = "/kaggle/working/b_tapt_100.zip"
run([sys.executable,"work/make_submission.py","--task","b",
     "--pred","work/runs/b_tapt_100/predictions.csv","--out",ZIP])
for f in ("work/tapt_100.log","work/b_tapt_100.log"):
    run(["cp",f,"/kaggle/working/"])
run(["cp","work/runs/b_tapt_100/predictions.csv","/kaggle/working/b_tapt_100_predictions.csv"])
run(["unzip","-l",ZIP])

new = pd.read_csv("work/runs/b_tapt_100/predictions.csv").set_index("id")["label"]
ref = pd.read_csv("submissions/b_tapt_5f/predictions.csv").set_index("id")["label"]
print(f"\nagrees with b_tapt_5f on {100*(new.reindex(ref.index) == ref).mean():.1f}% of test rows")
print("\npredicted distribution:")
print((100*new.value_counts(normalize=True)).round(1).to_string())
print("\nDownload b_tapt_100.zip from the Output tab and upload it to the Task B phase.")